In [43]:
from collections import defaultdict
from math import pi
from math import e
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [38]:
class GaussNB:
    def __init__(self):
        pass

    def group_by_class(self, data, target):
        target_map = defaultdict(list)
        n = len(data)
        for index in range(len(data)):
            features = data[index]
            if not features:
                continue
            x = features[target]
            target_map[x].append(features[:-1])  
        return dict(target_map)

    def mean(self, numbers):
        result = sum(numbers) / float(len(numbers))
        return result

    def dispersion(self, numbers):
        avg = self.mean(numbers)
        dispersion_list = []
        for num in numbers:
            dispersion = (num - avg) ** 2
            dispersion_list.append(dispersion)
        dispersion_sum = sum(dispersion_list)
        sample_count = float(len(numbers) - 1)
        var = dispersion_sum / sample_count
        return var ** .5

    def summarize(self, test_set):
        for feature in zip(*test_set):
            yield {
                'dispersion': self.dispersion(feature),
                'mean': self.mean(feature)
            }

    def aprior_prob(self, group, target, data):
        total = float(len(data))
        result = len(group[target]) / total
        return result

    def train(self, train_list, target):
        group = self.group_by_class(train_list, target)
        self.summaries = {}
        for target, features in group.items():
            self.summaries[target] = {
                'aprior_prob': self.aprior_prob(group, target, train_list),
                'summary': [i for i in self.summarize(features)],
            }
        return self.summaries

    def gauss_prob(self, x, mean, dispersion):
        variance = dispersion ** 2
        if variance == 0.0:
            variance = 0.0000000000000001
        exp_dispersion = (x - mean) ** 2
        exp_power = -exp_dispersion / (2 * variance)
        exponent = e ** exp_power
        multiplier = ((2 * pi) ** .5) * dispersion
        if multiplier == 0.0:
            multiplier = 0.000000000000001
        gauss_prob = exponent / multiplier
        return gauss_prob

    def total_prob(self, multiplied_probabilities):
        total_prob = sum(multiplied_probabilities.values())
        return total_prob

    def multiplied_probabilities(self, test_row):
        multiplied_probs = {}
        for target, features in self.summaries.items():
            total_features = len(features['summary'])
            likelihood = 1
            for index in range(total_features):
                feature = test_row[index]
                mean = features['summary'][index]['mean']
                dispersion = features['summary'][index]['dispersion']
                gauss_prob = self.gauss_prob(feature, mean, dispersion)
                likelihood *= gauss_prob
            aprior_prob = features['aprior_prob']
            multiplied_probs[target] = aprior_prob * likelihood
        return multiplied_probs

    def aposterior_probabilities(self, test_row):
        aposterior_probs = {}
        multiplied_probs = self.multiplied_probabilities(test_row)
        total_prob = self.total_prob(multiplied_probs)
        for target, mult_prob in multiplied_probs.items():
            aposterior_probs[target] = mult_prob / total_prob
        return aposterior_probs

    def get_map(self, test_row):
        aposterior_probs = self.aposterior_probabilities(test_row)
        map_prob = max(aposterior_probs, key=aposterior_probs.get)
        return map_prob

    def predict(self, test_set):
        map_probs = []
        for row in test_set:
            map_prob = self.get_map(row)
            map_probs.append(map_prob)
        return map_probs

    def accuracy(self, test_set, predicted):
        correct = 0
        actual = [item[-1] for item in test_set]
        for x, y in zip(actual, predicted):
            if x == y:
                correct += 1
        return correct / float(len(test_set)) * 100

In [44]:
class Kmean:
    def __init__(self):
        pass
    def train(self, training_data):
        self.training_data = training_data
    def eucledianDistance(self, test_data):
        dist = []
        for i in range(len(self.training_data)):
            sum_dist = 0
            for j in range(len(test_data)-1):
                sum_dist += (test_data[j] - self.training_data[i][j]) ** 2
            sum_dist = sum_dist ** .5
            dist.append({'class':self.training_data[i][-1], 'dist':sum_dist})
        return dist
    def predict(self, test_data, k):
        predicted = []
        for item in test_data:
            prediction = {}
            distances = self.eucledianDistance(item)
            distances = sorted(distances, key = lambda x:x['dist'])
            for i in range(k):
                prediction[distances[i]['class']] = prediction[distances[i]['class']] + 1 if prediction.get(distances[i]['class']) != None else 0 
            predicted.append(max(prediction, key = prediction.get))
        return predicted
    def accuracy(self, test_set, predicted):
        correct = 0
        actual = [item[-1] for item in test_set]
        for x, y in zip(actual, predicted):
            if x == y:
                correct += 1
        return correct / float(len(test_set)) * 100

In [13]:
import pandas as pd

irises = pd.read_csv('drive/MyDrive/data/iris.csv')

In [15]:
irises_classes = irises['species'].unique()
print(irises_classes)

['setosa' 'versicolor' 'virginica']


In [29]:
FILENAME = 'drive/MyDrive/data/iris.csv'
inputs = []
classes = {'setosa' : 1, 'versicolor' : 2, 'virginica' : 3}
with open(FILENAME) as file:
  line_number = 1
  for line in file:
    if line_number > 2:
      vars = line.split(',')
      input_list = [float(vars[i]) for i in range(4)]
      output_list = classes[vars[4][0:-1]]
      input_list.append(output_list)
      inputs.append(input_list)
    line_number = line_number + 1

print(inputs)

[[4.9, 3.0, 1.4, 0.2, 1], [4.7, 3.2, 1.3, 0.2, 1], [4.6, 3.1, 1.5, 0.2, 1], [5.0, 3.6, 1.4, 0.2, 1], [5.4, 3.9, 1.7, 0.4, 1], [4.6, 3.4, 1.4, 0.3, 1], [5.0, 3.4, 1.5, 0.2, 1], [4.4, 2.9, 1.4, 0.2, 1], [4.9, 3.1, 1.5, 0.1, 1], [5.4, 3.7, 1.5, 0.2, 1], [4.8, 3.4, 1.6, 0.2, 1], [4.8, 3.0, 1.4, 0.1, 1], [4.3, 3.0, 1.1, 0.1, 1], [5.8, 4.0, 1.2, 0.2, 1], [5.7, 4.4, 1.5, 0.4, 1], [5.4, 3.9, 1.3, 0.4, 1], [5.1, 3.5, 1.4, 0.3, 1], [5.7, 3.8, 1.7, 0.3, 1], [5.1, 3.8, 1.5, 0.3, 1], [5.4, 3.4, 1.7, 0.2, 1], [5.1, 3.7, 1.5, 0.4, 1], [4.6, 3.6, 1.0, 0.2, 1], [5.1, 3.3, 1.7, 0.5, 1], [4.8, 3.4, 1.9, 0.2, 1], [5.0, 3.0, 1.6, 0.2, 1], [5.0, 3.4, 1.6, 0.4, 1], [5.2, 3.5, 1.5, 0.2, 1], [5.2, 3.4, 1.4, 0.2, 1], [4.7, 3.2, 1.6, 0.2, 1], [4.8, 3.1, 1.6, 0.2, 1], [5.4, 3.4, 1.5, 0.4, 1], [5.2, 4.1, 1.5, 0.1, 1], [5.5, 4.2, 1.4, 0.2, 1], [4.9, 3.1, 1.5, 0.1, 1], [5.0, 3.2, 1.2, 0.2, 1], [5.5, 3.5, 1.3, 0.2, 1], [4.9, 3.1, 1.5, 0.1, 1], [4.4, 3.0, 1.3, 0.2, 1], [5.1, 3.4, 1.5, 0.2, 1], [5.0, 3.5, 1.3, 0.3, 1],

In [30]:
import random

random.shuffle(inputs)

In [46]:
kmean = Kmean()
kmean.train(inputs[:training_examples])
predicted = kmean.predict(inputs[training_examples:],3)
accuracy = kmean.accuracy(inputs[training_examples:],predicted)
print('================================================')
print('Точность метода k средних = ' + str(accuracy) + '%')

Точность метода k средних = 97.95918367346938%


In [40]:
bayess = GaussNB()
bayess.train(inputs[:training_examples],-1)
predicted = bayess.predict(test_inputs)
accuracy = bayess.accuracy(inputs[training_examples:], predicted)
print('================================================')
print('Точность наивного байесовского классификатора = ' + str(accuracy) + '%')

Точность наивного байесовского классификатора = 95.91836734693877%
